# NYC Taxi Trip Duration Prediction

## 1. Problem Statement

This project addresses a real-world machine learning problem focused on automating business processes. We will build a model to predict the total duration of taxi rides in New York City.

**Business Context:**
When you book a taxi from one point in NYC to another, how much should you pay? Taxi fares in the USA are calculated based on:
- A fixed base rate
- A time and distance-dependent tariff that varies by location

Trip duration depends on many factors such as:
- Origin and destination
- Time of day
- Weather conditions
- Traffic patterns

By developing an algorithm that accurately predicts trip duration, we can automatically forecast trip costs by applying the appropriate tariff rate.

Taxi services maintain massive datasets with historical trip information including pickup location, dropoff location, trip date, and duration. This data can be used to automatically predict trip duration using machine learning.

### Business Objective
Determine characteristics and use them to forecast the duration of taxi trips.

### Technical Objective
Build a machine learning model that predicts trip duration (in seconds) based on customer and trip characteristics. This is a **regression problem**.

### Key Project Goals
1. Aggregate and prepare data from multiple sources
2. Engineer features and identify the most predictive ones
3. Explore data and identify patterns
4. Train multiple models and select the best one
5. Design a prediction system for new data


## 2. Data Exploration and Feature Engineering

Let's start by exploring the provided data and augmenting it with external data sources to expand our dataset.

### Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy import stats
from sklearn import linear_model
from sklearn import preprocessing
from sklearn import model_selection
from sklearn import tree
from sklearn import ensemble
from sklearn import metrics
from sklearn import cluster
from sklearn import feature_selection

import warnings
warnings.simplefilter("ignore", UserWarning)

: 

In [ ]:
taxi_data = pd.read_csv("data/train.csv")
print('Train data shape: {}'.format(taxi_data.shape))
taxi_data.head()

### Data Overview and Feature Description

We have approximately 1.5 million trips described by 11 features, organized into several categories:

**Client and Vendor Data:**
- `id`: Unique trip identifier
- `vendor_id`: Taxi company identifier

**Temporal Features:**
- `pickup_datetime`: When the meter started
- `dropoff_datetime`: When the meter stopped

**Geographic Information:**
- `pickup_longitude`, `pickup_latitude`: Pickup location coordinates
- `dropoff_longitude`, `dropoff_latitude`: Dropoff location coordinates

**Other Features:**
- `passenger_count`: Number of passengers
- `store_and_fwd_flag`: Vehicle memory flag (Y/N)

**Target Variable:**
- `trip_duration`: Duration in seconds (what we're predicting)

In [ ]:
# Convert datetime to proper format
taxi_data['pickup_datetime'] = pd.to_datetime(taxi_data['pickup_datetime'], format='%Y-%m-%d %H:%M:%S')
taxi_data['pickup_datetime'].describe()

In [ ]:
# Check for missing values
taxi_data.isnull().sum()

In [ ]:
# Statistical summary
print(taxi_data['vendor_id'].value_counts())
print(taxi_data['passenger_count'].value_counts())
print(taxi_data['trip_duration'].describe())

### Temporal Feature Engineering

In [ ]:
def add_datetime_features(df):
    df['pickup_date'] = df['pickup_datetime'].dt.date
    df['pickup_hour'] = df['pickup_datetime'].dt.hour
    df['pickup_day_of_week'] = df['pickup_datetime'].dt.dayofweek
    return df

taxi_data = add_datetime_features(taxi_data)
print(taxi_data['pickup_day_of_week'].value_counts())
print(taxi_data['pickup_date'].value_counts().mean())

### Holiday Feature Engineering

In [ ]:
# Load holiday data
holiday_data = pd.read_csv('data/holiday_data.csv', sep=';')

# Add holiday features
def add_holiday_features(df, holiday_data):
    df['pickup_date'] = pd.to_datetime(df['pickup_date'])
    holiday_data['date'] = pd.to_datetime(holiday_data['date'])
    df = pd.merge(df, holiday_data, left_on='pickup_date', right_on='date', how='left')
    df['pickup_holiday'] = df['date'].apply(lambda x: 1 if pd.notnull(x) else 0)
    df.drop(['day', 'date', 'holiday'], axis=1, inplace=True)
    return df

taxi_data = add_holiday_features(taxi_data, holiday_data)
taxi_data[taxi_data['pickup_holiday'] == 1]['trip_duration'].describe()

### OSRM Routing Data Integration

In [ ]:
# Load OSRM data
osrm_data = pd.read_csv('data/osrm_data_train.csv')

# Add OSRM features
def add_osrm_features(df, osrm_data):
    osrm_data = osrm_data[['id', 'total_distance', 'total_travel_time', 'number_of_steps']]
    df = pd.merge(df, osrm_data, on='id', how='left')
    return df

taxi_data = add_osrm_features(taxi_data, osrm_data)
print(taxi_data['trip_duration'].median() - taxi_data['total_travel_time'].median())

### Geographic Features

**Haversine Distance:**
Calculates the great-circle distance between two geographic points on Earth.

$$h = 2R \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta lat}{2}\right) + \cos(lat_1)\cos(lat_2)\sin^2\left(\frac{\Delta lon}{2}\right)}\right)$$

In [ ]:
# Using the Haversine formula to calculate the distance between pickup and dropoff points
def get_haversine_distance(lat1, lng1, lat2, lng2):
    lat1, lng1, lat2, lng2 = map(np.radians, (lat1, lng1, lat2, lng2))
    EARTH_RADIUS = 6371 
    lat_delta = lat2 - lat1
    lng_delta = lng2 - lng1
    d = np.sin(lat_delta * 0.5) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(lng_delta * 0.5) ** 2
    h = 2 * EARTH_RADIUS * np.arcsin(np.sqrt(d))
    return h

# Calculate the angle between pickup and dropoff points
def get_angle_direction(lat1, lng1, lat2, lng2):
    lat1, lng1, lat2, lng2 = map(np.radians, (lat1, lng1, lat2, lng2))
    lng_delta_rad = lng2 - lng1
    y = np.sin(lng_delta_rad) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(lng_delta_rad)
    alpha = np.degrees(np.arctan2(y, x))
    return alpha

# Add geographical features
def add_geographical_features(df):
    df['haversine_distance'] = get_haversine_distance(df['pickup_latitude'], df['pickup_longitude'], 
                                                      df['dropoff_latitude'], df['dropoff_longitude'])
    df['direction'] = get_angle_direction(df['pickup_latitude'], df['pickup_longitude'], 
                                         df['dropoff_latitude'], df['dropoff_longitude'])
    return df

taxi_data = add_geographical_features(taxi_data)
taxi_data['haversine_distance'].describe()

### Geographic Clustering

In [ ]:
# Create geographic clusters using KMeans
coords = np.hstack((taxi_data[['pickup_latitude', 'pickup_longitude']],
                    taxi_data[['dropoff_latitude', 'dropoff_longitude']]))
kmeans = cluster.KMeans(n_clusters=10, random_state=42)
kmeans.fit(coords)

def add_cluster_features(df, kmeans):
    df['geo_cluster'] = kmeans.predict(coords)
    return df

taxi_data = add_cluster_features(taxi_data, kmeans)
taxi_data['geo_cluster'].value_counts()

### Weather Feature Integration

In [ ]:
# Load and add weather data
weather_data = pd.read_csv('data/weather_data.csv')

def add_weather_features(df, weather_data):
    weather_data['time'] = pd.to_datetime(weather_data['time'])
    weather_data['date'] = weather_data['time'].dt.date
    weather_data['hour'] = weather_data['time'].dt.hour
    weather_data = weather_data[['date', 'hour', 'temperature', 'visibility', 'wind speed', 'precip', 'events']]
    df['pickup_hour'] = df['pickup_datetime'].dt.hour
    df['pickup_date'] = df['pickup_datetime'].dt.date
    df = pd.merge(df, weather_data, left_on=['pickup_date', 'pickup_hour'], right_on=['date', 'hour'], how='left')
    df.drop(['date', 'hour'], axis=1, inplace=True)
    return df

taxi_data = add_weather_features(taxi_data, weather_data)
print(taxi_data['events'].value_counts())
print(taxi_data['temperature'].isnull().sum() / taxi_data.shape[0] * 100)

### Missing Value Imputation

In [ ]:
def fill_null_weather_data(df):
    for col in ['temperature', 'visibility', 'wind speed', 'precip']:
        df[col] = df[col].fillna(df.groupby('pickup_date')[col].transform('median'))
    df['events'] = df['events'].fillna('None')
    for col in ['total_distance', 'total_travel_time', 'number_of_steps']:
        df[col] = df[col].fillna(df[col].median())
    return df

taxi_data = fill_null_weather_data(taxi_data)
taxi_data['temperature'].median()

### Outlier Detection and Removal

We identify and remove obvious outliers:
- Trips longer than 24 hours
- Trips with average speed > 300 km/h (teleportation)

In [ ]:
# Visualize the relationship between average speed and trip duration
avg_speed = taxi_data['total_distance'] / taxi_data['trip_duration'] * 3.6
fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(x=avg_speed.index, y=avg_speed, ax=ax)
ax.set_xlabel('Index')
ax.set_ylabel('Average speed')
ax.set_title('Average speed for each trip')
plt.show()

In [ ]:
# Remove outliers based on trip duration and average speed
taxi_data.drop(taxi_data[taxi_data['trip_duration'] / 3600 >= 24].index, inplace=True)
mask = avg_speed > 300
taxi_data.drop(taxi_data[mask].index, inplace=True)
taxi_data.info()

## 3. Exploratory Data Analysis (EDA)

In this section, we explore patterns in the data and create visualizations.

### Target Variable Distribution

In [ ]:
# Log transform trip duration (required for RMSLE metric)
taxi_data['trip_duration_log'] = np.log(taxi_data['trip_duration']+1)

# Visualize distribution
fig, ax = plt.subplots(figsize=(10, 5), nrows=1, ncols=2)
sns.histplot(taxi_data['trip_duration_log'], bins=50, kde=True, ax=ax[0])
sns.boxplot(x=taxi_data['trip_duration_log'], ax=ax[1])
ax[0].set_title('Distribution of Log Trip Duration')
ax[1].set_title('Boxplot of Log Trip Duration')
plt.show()

### Comparison of visual inspection and statical test

In [ ]:
# Normality test
import scipy.stats
res = scipy.stats.normaltest(taxi_data['trip_duration_log'])
print(f"Normality Test (D'Agostino):")
print(f"p-value: {res.pvalue:.4f}")
print(f"Is distribution normal (alpha=0.05)? {res.pvalue > 0.05}")

### Temporal Patterns

In [ ]:
# Analyze by hour
fig, ax = plt.subplots(figsize=(10, 14), nrows=2, ncols=1)

sns.histplot(taxi_data['pickup_hour'], bins=24, ax=ax[0], kde=False)
ax[0].set_title('Distribution of Trips by Hour')
ax[0].set_xlabel('Pickup Hour')
ax[0].set_ylabel('Number of Trips')

duration_by_hour = taxi_data.groupby('pickup_hour')['trip_duration_log'].median()
sns.lineplot(x=duration_by_hour.index, y=duration_by_hour, ax=ax[1], marker='o')
ax[1].set_title('Median Trip Duration by Hour')
ax[1].set_xlabel('Pickup Hour')
ax[1].set_ylabel('Median Log Duration')
ax[1].set_xticks(np.arange(0, 24, 1))

plt.tight_layout()
plt.show()

# Find peak and minimum hours
min_hour = taxi_data['pickup_hour'].value_counts().idxmin()
max_duration_hour = duration_by_hour.idxmax()
print(f"Minimum taxi demand at: {min_hour}:00")
print(f"Peak trip duration at: {max_duration_hour}:00")

### Day of Week Patterns

In [ ]:
# Map day numbers to names
day_names = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 
             4: 'Friday', 5: 'Saturday', 6: 'Sunday'}

fig, ax = plt.subplots(figsize=(10, 14), nrows=2, ncols=1)

day_counts = taxi_data['pickup_day_of_week'].value_counts().sort_index()
sns.barplot(x=[day_names[i] for i in day_counts.index], y=day_counts.values, ax=ax[0])
ax[0].set_title('Distribution of Trips by Day of Week')
ax[0].set_xlabel('Day of Week')
ax[0].set_ylabel('Number of Trips')

duration_by_day = taxi_data.groupby('pickup_day_of_week')['trip_duration_log'].median()
sns.lineplot(x=[day_names[i] for i in duration_by_day.index], y=duration_by_day.values, ax=ax[1], marker='o')
ax[1].set_title('Median Trip Duration by Day of Week')
ax[1].set_xlabel('Day of Week')
ax[1].set_ylabel('Median Log Duration')

plt.setp(ax[0].xaxis.get_majorticklabels(), rotation=45)
plt.setp(ax[1].xaxis.get_majorticklabels(), rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze by hour and day of week
table = pd.pivot_table(taxi_data, 
                    values='trip_duration', 
                    index='pickup_hour', 
                    columns='pickup_day_of_week', 
                    aggfunc='median')
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(table, ax=ax, cmap='coolwarm', annot=True, fmt=".1f")
ax.set_title('Median trip duration by pickup hour and day of week')

### Geographic Analysis

In [ ]:
# City borders for visualization
city_long_border = (-74.03, -73.75)
city_lat_border = (40.63, 40.85)

# Visualize clusters on the map
def visualize_clusters(col_1, col_2, title, ax):
    sns.scatterplot(x=col_1, y=col_2, hue=taxi_data['geo_cluster'], palette='Set1', ax=ax, s=5)
    ax.set_xlim(city_long_border)
    ax.set_ylim(city_lat_border)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(title)

fig, ax = plt.subplots(figsize=(10, 10), nrows=2, ncols=1)
vis_1 = visualize_clusters(taxi_data['pickup_longitude'], taxi_data['pickup_latitude'], 'Pickup locations', ax[0])
vis_2 = visualize_clusters(taxi_data['dropoff_longitude'], taxi_data['dropoff_latitude'], 'Dropoff locations', ax[1])

### The other graphs

In [ ]:
# Visualize the relationship between number of steps and trip duration
fig, ax = plt.subplots(figsize=(10, 5), nrows=1, ncols=2)
sns.scatterplot(x=taxi_data['number_of_steps'], y=taxi_data['trip_duration_log'], hue=taxi_data['geo_cluster'], palette='Set1', ax=ax[0])
sns.boxplot(x=taxi_data['passenger_count'], y=taxi_data['trip_duration_log'], ax=ax[1])

ax[0].set_title('Trip Duration vs Number of Steps')
ax[0].set_xlabel('Number of steps')
ax[0].set_ylabel('Log of trip duration')

ax[1].set_title('Trip Duration vs Passenger Count')
ax[1].set_xlabel('Passenger count')
ax[1].set_ylabel('Log of trip duration')

In [ ]:
# Analyze by vendor_id
fig, ax = plt.subplots(figsize=(10, 5), nrows=1, ncols=2)
for group in taxi_data['vendor_id'].unique():
    sns.histplot(taxi_data[taxi_data['vendor_id'] == group]['trip_duration_log'], bins=50, kde=True, ax=ax[0])
    sns.boxplot(x=taxi_data[taxi_data['vendor_id'] == group]['trip_duration_log'], ax=ax[1], label='vendor_id: {}'.format(group))
ax[0].set_title('Distribution of Log Trip Duration by Vendor')
ax[0].set_xlabel('Log Trip Duration')
ax[0].set_ylabel('Number of Trips')
ax[1].set_xlabel('Log Trip Duration')
plt.show()

In [ ]:
# Analyze by store and forward flag
fig, ax = plt.subplots(figsize=(10, 10), nrows=2, ncols=2)
mask_y = taxi_data['store_and_fwd_flag'] == 'Y'
mask_n = taxi_data['store_and_fwd_flag'] == 'N'

sns.histplot(taxi_data[mask_y]['trip_duration_log'], bins=50, kde=True, ax=ax[0, 0]);
sns.boxplot(x=taxi_data[mask_y]['trip_duration_log'], ax=ax[0, 1]);
sns.histplot(taxi_data[mask_n]['trip_duration_log'], bins=50, kde=True, ax=ax[1, 0]);
sns.boxplot(x=taxi_data[mask_n]['trip_duration_log'], ax=ax[1, 1]);
ax[0, 0].set_title('Store and Forward Flag')
ax[1, 0].set_title('No Store and Forward Flag')


## 4. Feature Selection and Preprocessing

The following steps prepare features, split the dataset, train regression models, and evaluate prediction quality.


### Remove Non-Informative Features

In [ ]:
train_data = taxi_data.copy()

# Remove features with data leakage and redundant features
train_data = train_data.drop(['id', 'dropoff_datetime'], axis=1)

# Remove datetime features (we've already extracted temporal features)
train_data = train_data.drop(['pickup_datetime', 'pickup_date'], axis=1)

print(f"Remaining features: {train_data.shape[1]}")
print(f"Columns: {train_data.columns.tolist()}")

### Encode Binary Features

In [ ]:
train_data['vendor_id'] = train_data['vendor_id'].apply(lambda x: 0 if x == 1 else 1)
train_data['store_and_fwd_flag'] = train_data['store_and_fwd_flag'].apply(lambda x: 0 if x == 'N' else 1)

### One-Hot Encode and Combine Categorical Features

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Apply one-hot encoding
one_hot_encoder = OneHotEncoder(drop='first', handle_unknown='ignore')
columns_to_change = ['pickup_day_of_week', 'events', 'geo_cluster']
data_onehot = one_hot_encoder.fit_transform(train_data[columns_to_change])

# Convert to DataFrame
column_names = one_hot_encoder.get_feature_names_out()
data_onehot = pd.DataFrame.sparse.from_spmatrix(data_onehot, columns=column_names)

# Combine processed features
train_data = pd.concat(
    [train_data.reset_index(drop=True).drop(columns_to_change, axis=1), data_onehot],
    axis=1
)

print(f"Final dataset shape: {train_data.shape}")
print(f"Features: {train_data.shape[1]}")


### Prepare Training Data

In [ ]:
# Separate features and target
X = train_data.drop(['trip_duration', 'trip_duration_log'], axis=1)
y = train_data['trip_duration']
y_log = train_data['trip_duration_log']

### Feature Selection with SelectKBest

In [ ]:
# Split data
X_train, X_valid, y_train_log, y_valid_log = model_selection.train_test_split(
    X, y_log, test_size=0.33, random_state=42
)

# Feature selection
selectKBest = feature_selection.SelectKBest(feature_selection.f_regression, k=25)
selectKBest.fit(X_train, y_train_log)
selected_features = X_train.columns[selectKBest.get_support()]

print(f"Selected {len(selected_features)} features:")
for i, feature in enumerate(selected_features, 1):
    print(f"{i}. {feature}")

### Feature Scaling

In [ ]:
# Scale features for models that require it
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train[selected_features])
X_valid_scaled = scaler.transform(X_valid[selected_features])
print(X_valid_scaled[:, 0].mean())

## 5. Model Development and Evaluation

### Evaluation Metric: RMSLE

Root Mean Squared Logarithmic Error (RMSLE):

$$RMSLE = \sqrt{\frac{1}{n}\sum_{i=1}^n(\log(y_i+1)-\log(\hat{y_i}+1))^2}$$


Target feature has already been log-transformed, so calculation formula will be:
$$z_i=log(y_i+1),$$
$RMSLE = \sqrt{\frac{1}{n}\sum_{i=1}^n(z_i-\hat{z_i})^2}=\sqrt{MSE(z_i,\hat{z_i})}$

In [ ]:
from sklearn.linear_model import LinearRegression

# Train model
model_lr = LinearRegression()
model_lr.fit(X_train_scaled, y_train_log)

# Predictions
y_train_lr_pred = model_lr.predict(X_train_scaled)
y_valid_lr_pred = model_lr.predict(X_valid_scaled)

# Evaluation
train_rmsle_lr = np.sqrt(metrics.mean_squared_error(y_train_log, y_train_lr_pred))
valid_rmsle_lr = np.sqrt(metrics.mean_squared_error(y_valid_log, y_valid_lr_pred))

print("Linear Regression Results:")
print(f"Train RMSLE: {train_rmsle_lr:.3f}")
print(f"Valid RMSLE: {valid_rmsle_lr:.3f}")

### Decision Tree Regression

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Find optimal depth
train_errors = []
valid_errors = []
max_depths = range(7, 20)

for max_depth in max_depths:
    dt = DecisionTreeRegressor(max_depth=max_depth, random_state=42)
    dt.fit(X_train_scaled, y_train_log)
    
    y_train_dt_pred = dt.predict(X_train_scaled)
    y_valid_dt_pred = dt.predict(X_valid_scaled)
    
    train_errors.append(np.sqrt(metrics.mean_squared_error(y_train_log, y_train_dt_pred)))
    valid_errors.append(np.sqrt(metrics.mean_squared_error(y_valid_log, y_valid_dt_pred)))

# Plot results
plt.figure(figsize=(10, 6))
plt.plot(max_depths, train_errors, label='Train RMSLE', marker='o')
plt.plot(max_depths, valid_errors, label='Valid RMSLE', marker='o')
plt.xlabel('Max Depth')
plt.ylabel('RMSLE')
plt.title('Decision Tree: Impact of Max Depth')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Find optimal depth
optimal_depth = max_depths[np.argmin(valid_errors)]
train_rmsle_dt = train_errors[np.argmin(valid_errors)]
valid_rmsle_dt = valid_errors[np.argmin(valid_errors)]
print(f"Optimal max depth: {optimal_depth}")
print(f"Best valid RMSLE: {valid_rmsle_dt:.3f}")

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Train Random Forest
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    criterion='squared_error',
    min_samples_split=20,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_scaled, y_train_log)

# Predictions
y_train_rf_pred = rf.predict(X_train_scaled)
y_valid_rf_pred = rf.predict(X_valid_scaled)

# Evaluation
train_rmsle_rf = np.sqrt(metrics.mean_squared_error(y_train_log, y_train_rf_pred))
valid_rmsle_rf = np.sqrt(metrics.mean_squared_error(y_valid_log, y_valid_rf_pred))

print("Random Forest Results:")
print(f"Train RMSLE: {train_rmsle_rf:.3f}")
print(f"Valid RMSLE: {valid_rmsle_rf:.3f}")

### Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# Train Gradient Boosting
gbr = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.5,
    min_samples_split=30,
    random_state=42
)

gbr.fit(X_train_scaled, y_train_log)

# Predictions
y_train_gbr_pred = gbr.predict(X_train_scaled)
y_valid_gbr_pred = gbr.predict(X_valid_scaled)

# Evaluation
train_rmsle_gbr = np.sqrt(metrics.mean_squared_error(y_train_log, y_train_gbr_pred))
valid_rmsle_gbr = np.sqrt(metrics.mean_squared_error(y_valid_log, y_valid_gbr_pred))

print("Gradient Boosting Results:")
print(f"Train RMSLE: {train_rmsle_gbr:.3f}")
print(f"Valid RMSLE: {valid_rmsle_gbr:.3f}")

### XGBoost model

In [ ]:
import xgboost as xgb

# Prepare data for XGBoost
selected_features = selected_features.tolist()
dtrain = xgb.DMatrix(X_train_scaled, label=y_train_log, feature_names=selected_features)
dvalid = xgb.DMatrix(X_valid_scaled, label=y_valid_log, feature_names=selected_features)

# Set XGBoost parameters
xgb_pars = {'min_child_weight': 20, 
            'eta': 0.1, 
            'colsample_bytree': 0.9, 
            'max_depth': 6, 
            'subsample': 0.9, 
            'lambda': 1, 
            'nthread': -1, 
            'booster' : 'gbtree', 
            'eval_metric': 'rmse', 
            'objective': 'reg:squarederror'
           }
watchlist = [(dtrain, 'train'), (dvalid, 'valid')]

# Train XGBoost model
model = xgb.train(
    params=xgb_pars, 
    dtrain=dtrain, 
    num_boost_round=300, 
    evals=watchlist, 
    early_stopping_rounds=20, 
    maximize=False,
    verbose_eval=10
)
# Predictions
y_train_xgb_pred = model.predict(dtrain)
y_valid_xgb_pred = model.predict(dvalid)

# Evaluation
train_rmsle_xgb = np.sqrt(metrics.mean_squared_error(y_train_log, y_train_xgb_pred))
valid_rmsle_xgb = np.sqrt(metrics.mean_squared_error(y_valid_log, y_valid_xgb_pred))

print("XGBoost Results:")
print(f"Train RMSLE: {train_rmsle_xgb:.3f}")
print(f"Valid RMSLE: {valid_rmsle_xgb:.3f}")

In [ ]:
# Visualize feature importance
fig, ax = plt.subplots(figsize = (15,15))
xgb.plot_importance(model, ax = ax, height=0.5)

## 6. Model Comparison

In [ ]:
# Compare all models
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosting', 'XGBoost'],
    'Train RMSLE': [train_rmsle_lr, train_rmsle_dt, train_rmsle_rf, train_rmsle_gbr, train_rmsle_xgb],
    'Valid RMSLE': [valid_rmsle_lr, valid_rmsle_dt, valid_rmsle_rf, valid_rmsle_gbr, valid_rmsle_xgb]
})

comparison['Overfitting Gap'] = comparison['Train RMSLE'] - comparison['Valid RMSLE']

print("\nModel Performance Comparison:")
print(comparison.to_string())

best_model_idx = comparison['Valid RMSLE'].idxmin()
best_model_name = comparison.loc[best_model_idx, 'Model']
best_rmsle = comparison.loc[best_model_idx, 'Valid RMSLE']

print(f"\n✓ Best Model: {best_model_name} (Valid RMSLE: {best_rmsle:.3f})")

### Feature Importance Analysis

In [ ]:
# Feature importance from best model (Gradient Boosting)
feature_importance = pd.DataFrame({
    'Feature': selected_features,
    'Importance': gbr.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance.head(15), x='Importance', y='Feature')
plt.title('Top 15 Most Important Features (Gradient Boosting)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print("\nTop 10 Most Important Features:")
for idx, row in feature_importance.head(10).iterrows():
    print(f"{row['Feature']}: {row['Importance']:.4f}")

### Model Performance Metrics

In [ ]:
# Calculate additional metrics
y_valid_actual = np.exp(y_valid_log) - 1  # Convert from log scale
y_valid_gbr_actual = np.exp(y_valid_gbr_pred) - 1  # Convert from log scale

# Median Absolute Error (in minutes)
mae = metrics.median_absolute_error(y_valid_actual, y_valid_gbr_actual)
mae_minutes = mae / 60

# Mean Absolute Error
mean_ae = metrics.mean_absolute_error(y_valid_actual, y_valid_gbr_actual)

# R² Score
r2 = metrics.r2_score(y_valid_log, y_valid_gbr_pred)

print("Detailed Performance Metrics (Gradient Boosting):")
print(f"Valid RMSLE: {valid_rmsle_gbr:.3f}")
print(f"Median Absolute Error: {mae:.0f} seconds ({mae_minutes:.1f} minutes)")
print(f"Mean Absolute Error: {mean_ae:.0f} seconds")
print(f"R² Score (log scale): {r2:.4f}")

## 7. Test Set Predictions and Submission

Now we generate predictions for the test set and prepare submission for Kaggle.

In [ ]:
# Load test data
test_data = pd.read_csv("data/test_data.csv")
osrm_data_test = pd.read_csv("data/osrm_data_test.csv")
test_id = test_data['id']
print(f"Test data shape: {test_data.shape}")
print(f"OSRM test data shape: {osrm_data_test.shape}")

In [ ]:
# Apply same preprocessing as training data
test_data['pickup_datetime']=pd.to_datetime(test_data['pickup_datetime'],format='%Y-%m-%d %H:%M:%S')
test_data = add_datetime_features(test_data)
test_data = add_holiday_features(test_data, holiday_data)
test_data = add_osrm_features(test_data, osrm_data_test)
test_data = add_geographical_features(test_data)

# Geographic clustering
coords_test = np.hstack((test_data[['pickup_latitude', 'pickup_longitude']],
                    test_data[['dropoff_latitude', 'dropoff_longitude']]))
test_data['geo_cluster'] = kmeans.predict(coords_test)

# Weather features
test_data = add_weather_features(test_data, weather_data)
test_data = fill_null_weather_data(test_data)

# Encode categorical features
test_data['vendor_id'] = test_data['vendor_id'].apply(lambda x: 0 if x == 1 else 1)
test_data['store_and_fwd_flag'] = test_data['store_and_fwd_flag'].apply(lambda x: 0 if x == 'N' else 1)

# One-hot encode test data using the same encoder
test_data_onehot = one_hot_encoder.fit_transform(test_data[columns_to_change]).toarray()
column_names = one_hot_encoder.get_feature_names_out(columns_to_change)
test_data_onehot = pd.DataFrame(test_data_onehot, columns=column_names)

# Combine features
test_data = pd.concat(
    [test_data.reset_index(drop=True).drop(columns_to_change, axis=1), 
     test_data_onehot], 
    axis=1
)

# Select same features
X_test = test_data[selected_features]
X_test_scaled = scaler.transform(X_test)
print('Shape of data: {}'.format(X_test_scaled.shape))

### Generate predictions

In [ ]:
# Predict on test data
y_test_pred_log = gbr.predict(X_test_scaled)
y_test_pred = np.exp(y_test_pred_log) - 1
print(f"Predictions generated: {len(y_test_pred)}")
print(f"\nPrediction statistics:")
print(f"Mean: {y_test_pred.mean():.0f} seconds")
print(f"Median: {np.median(y_test_pred):.0f} seconds")
print(f"Min: {y_test_pred.min():.0f} seconds")
print(f"Max: {y_test_pred.max():.0f} seconds")

In [ ]:
# Create submission file
submission = pd.DataFrame({'id': test_id, 'trip_duration': y_test_pred})
submission.to_csv('data/submission_gb.csv', index=False)

# Project Conclusion

This notebook demonstrates a complete production-style machine learning workflow for trip duration prediction.

## Results
- Built a full preprocessing pipeline
- Engineered temporal and geospatial features
- Trained regression models
- Evaluated prediction accuracy
- Interpreted feature importance

In [ ]:
print(f"\n" + "="*50)
print("PROJECT SUMMARY")
print("="*50)
print(f"\n✓ Dataset: 1.5M+ NYC taxi trips")
print(f"✓ Features Engineered: 78 (after feature selection: 25)")
print(f"\n✓ Best Model: Gradient Boosting Regressor")
print(f"  - Training RMSLE: {train_rmsle_gbr:.3f}")
print(f"  - Validation RMSLE: {valid_rmsle_gbr:.3f}")
print(f"  - Median Prediction Error: {mae_minutes:.1f} minutes")
print(f"\n✓ Key Insights:")
print(f"  - Top 3 predictive features: {', '.join(feature_importance.head(3)['Feature'].tolist())}")
print(f"  - Rush hours (longest trips): 8-10 AM, 5-7 PM")
print(f"  - Minimum demand: 2-5 AM")
print(f"\n✓ Submission: {submission.shape[0]} predictions")
print(f"  - File: data/submissions/submission_gb.csv")
print("\n" + "="*50)

In [ ]:
# Save model for future use
import joblib
import os

# Create models dictionary
models_dict = {
    'best_model': gbr,
    'scaler': scaler,
    'encoder': one_hot_encoder,
    'selected_features': selected_features,
    'kmeans': kmeans
}

# Save models
os.makedirs('../models', exist_ok=True)
joblib.dump(models_dict, '../models/trained_models.pkl')
print("✓ Models saved to models/trained_models.pkl")